# 03. 신뢰 가능한 에이전트를 위한 5가지 사고 도구

> 이 사고틀을 산업에서는 *harness engineering* 이라 부르기도 합니다. 이 강의에서는 용어보다 **5가지 도구 자체**에 집중해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **Agent = Model + 주변 코드** 라는 관점을 이해하고, "모델 자체를 제외한 모든 것"이 왜 엔지니어링 대상인지 설명할 수 있어요
2. **5가지 신뢰성 원칙**(Constrain · Inform · Verify · Correct · HITL)을 구분하고, 각 원칙이 어떤 실패 모드를 막는지 설명할 수 있어요
3. 각 원칙을 LangChain V1 / LangGraph / Deep Agents의 **구체적 컴포넌트**(`create_agent`, middleware, hook, `interrupt`)에 매핑할 수 있어요
4. 실제 산업 사례(Anthropic 3-agent 구성, MCP 도구 다이어트, context anxiety)에서 5가지 원칙이 어떻게 적용되는지 분석할 수 있어요

## 사전 지식

- 02-Design-Principles 노트북에서 다룬 5단계 방법론, 노드 분해, 에러 유형, `Command`, `interrupt()` 규칙
- `StateGraph`, `add_messages` reducer, 체크포인터의 기본 개념
- `create_agent` (LangChain V1)의 기본 사용법


## 시작하기 전에 — 용어에 대한 짧은 메모

이 노트북에서 다루는 5가지 도구(Constrain·Inform·Verify·Correct·HITL)는 2026년 산업에서 *harness engineering* 이라는 이름으로 정리되고 있습니다. 다만 이 강의의 정체성은 "에이전트 구축"이지 "하네스 마스터하기"가 아닙니다. 5가지 도구는 **이후 챕터에서 만들 LangGraph 컴포넌트(미들웨어·메모리·MCP·서브에이전트)를 왜·언제 조합할지** 판단하는 사고 프레임으로만 활용하세요. 1-2년 후 용어가 바뀌어도 도구 자체는 살아남습니다.


## 환경 설정


In [ ]:
# 환경 변수 로드 (.env 파일에서 API 키를 읽어와요)
from dotenv import load_dotenv
load_dotenv()


In [ ]:
# LangSmith 추적 설정 (학습 중 그래프 실행 과정을 추적할 수 있어요)
import os
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-Part03-ReliabilityEngineering"


In [ ]:
# 노트북 전체에서 사용할 모델
from langchain.chat_models import init_chat_model

# 비용 효율과 학생 접근성을 위해 gpt-4o-mini를 기본으로 사용해요
model = init_chat_model("openai:gpt-4o-mini")
# 모델 초기화 완료


## 1. Agent = Model + 주변 코드

에이전트를 만들다 보면 곧 깨닫게 돼요. **모델 자체보다 모델 주변의 코드가 훨씬 더 중요하다**는 사실을요.
이 "주변 코드" 전체를 묶어서 부르는 이름이 여러 개 있는데, 최근 산업에서는 **harness**(하네스)라는 단어가 자주 쓰여요. 영어로 *말이나 낙하산을 안전하게 묶어주는 장구*를 의미해요. 이름 자체보다는 그 안에 들어가는 도구들이 중요해요.

> Martin Fowler — *"A harness is everything in an AI agent except the model itself."*
> ([martinfowler.com](https://martinfowler.com/articles/harness-engineering.html))

즉, 시스템 프롬프트, 도구 정의, 메모리, 라우팅 로직, 검증 코드, 사람 개입 지점, 로깅, 재시도 정책 — 이 모든 것이 모델 주변의 코드예요.
모델은 **확률적으로 작동하는 부정확한 추론 엔진**이고, 이 주변 코드가 그 부정확함을 **제품 수준의 신뢰성**으로 바꿔주는 골격이에요.

> Anthropic — *"The harness imposes structure on the model's tendency to act imprecisely."*
> ([anthropic.com/engineering/harness-design-long-running-apps](https://www.anthropic.com/engineering/harness-design-long-running-apps))


### 시각적으로 정리하면

```mermaid
flowchart LR
    INPUT(["사용자 입력"]) --> H["주변 코드<br/>(prompts · tools · memory ·<br/>verification · routing · HITL)"]
    H --> M["Model<br/>(LLM 추론)"]
    M --> H
    H --> OUT(["신뢰 가능한 출력"])

    classDef harness fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef model fill:#cce5ff,stroke:#007bff,color:#004085
    class H harness
    class M model
```

### 관점의 이동

- **Prompt Engineering** 시대: 모델 한 번 호출, 프롬프트만 잘 짜면 끝
- **Context Engineering** 시대: 모델에게 *무엇을* 보여줄지(검색·요약·필터링)가 핵심
- **지금**: 모델을 *어떻게 감싸서* 자율적으로 안전하게 일하게 만들지가 핵심 (이걸 *harness engineering* 이라고 부르는 사람도 있어요)

장기 실행 에이전트(coding agent, deep research, customer support agent)가 등장하면서, **모델 외부의 골격이 곧 제품의 품질**이 되었어요. 용어에 집착하지 말고, 그 안에 들어가는 5가지 도구에 집중하세요.


## 2. 5가지 사고 도구

신뢰 가능한 에이전트를 만드는 주변 코드는 결국 다음 5가지 일을 해요. 외워두면 어떤 에이전트 시스템을 봐도 분해할 수 있어요.

```mermaid
flowchart TB
    subgraph H["신뢰 가능한 에이전트의 5가지 사고 도구"]
        C["Constrain<br/>할 수 있는 일을 제한"]
        I["Inform<br/>무엇을 해야 하는지 알린다"]
        V["Verify<br/>결과를 검증한다"]
        R["Correct<br/>실패를 자체 수정한다"]
        HL["HITL<br/>고위험 지점에 사람을 끼워 넣는다"]
    end

    classDef principle fill:#d4edda,stroke:#28a745,color:#155724
    class C,I,V,R,HL principle
```


### 5가지 도구 한눈에 보기

| 원칙 | 한 줄 정의 | 막아주는 실패 |
|---|---|---|
| **Constrain** | 도구·횟수·범위를 좁힌다 | 무한 루프, 과도한 비용, 위험 행동 |
| **Inform** | 컨텍스트를 동적으로 주입한다 | "그걸 어떻게 알아요?" 류의 환각 |
| **Verify** | LLM 출력을 검증한다 | 형식 오류, 사실 오류, 정책 위반 |
| **Correct** | 실패를 받아 다시 시도한다 | 일회성 오류로 인한 전체 중단 |
| **HITL** | 사람이 개입한다 | 되돌릴 수 없는 행동의 사고 |

> HumanLayer — *"Anytime the agent makes a mistake, engineer a solution so it never makes that mistake again."*

에이전트의 실수를 **모델 탓으로 돌리지 않고**, 주변 코드를 한 겹 더 두르는 태도 — 이게 5가지 도구의 공통된 사고방식이에요.
이제 각 원칙을 5분짜리 미니 데모로 하나씩 만나볼게요.


## 3. Constrain — 할 수 있는 일을 제한한다

에이전트가 도구를 무한히 호출하면 비용이 폭발하고, 무한 루프에 빠질 수 있어요.
LangChain V1의 `ToolCallLimitMiddleware`는 thread/run 단위로 도구 호출 횟수를 제한해요.

> 일반 원칙: **"권한은 최소로, 한도는 명시적으로."** 도구 자체뿐 아니라 *몇 번까지* 쓸 수 있는지도 주변 코드에서 미리 정해줘야 해요.


**예고편**: `ToolCallLimitMiddleware`는 **06_Middleware/04-Prebuilt-Middleware**에서 자세히 다룹니다. 여기서는 "할 수 있는 일을 제한한다"는 Constrain 원칙이 코드로 어떻게 표현되는지 감만 잡으세요.


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain_core.tools import tool

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Constrain 데모: ToolCallLimitMiddleware로 도구 호출 횟수를 제한
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 4. Inform — 무엇을 해야 하는지 알린다

에이전트가 사용자 이름을 모르면 "고객님"이라고 부르고, 사용자 선호를 모르면 평균값으로 답해요.
**필요한 컨텍스트를 매 호출마다 동적으로 주입**해야 해요. LangChain V1의 `@dynamic_prompt` 데코레이터가 이 역할을 해요.

> 데이터를 **상태로 들고 있는 것과 모델에 보여주는 것은 다른 일**이에요. Inform 원칙은 후자에 관한 거예요.


**예고편**: `@dynamic_prompt`는 **06_Middleware/03-Context-Engineering** 및 **07_Memory**에서 자세히 다룹니다. 여기서는 "컨텍스트를 동적으로 주입한다"는 Inform 원칙이 코드로 어떻게 표현되는지 감만 잡으세요.


In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langgraph.store.memory import InMemoryStore

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Inform 데모: Store에 저장된 사용자 정보를 dynamic_prompt로 주입
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. Verify — 결과를 검증한다

LLM은 종종 그럴듯하지만 **정책에 어긋나는** 답을 내놓아요. `@after_model` 훅으로 모델 출력 직후에 **LLM-as-judge** 검증을 끼워 넣을 수 있어요.

> Verify는 단지 형식 검증(JSON 파싱)만이 아니에요. **사실성·안전성·톤·정책 준수**까지 포함해요.


**예고편**: `@after_model` 검증 패턴은 **12_Testing/03**에서 자세히 다룹니다. 여기서는 "결과를 검증한다"는 Verify 원칙이 코드로 어떻게 표현되는지 감만 잡으세요.


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import after_model, AgentState
from langchain.chat_models import init_chat_model
from langgraph.runtime import Runtime

judge = init_chat_model("openai:gpt-4o-mini")

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Verify 데모: after_model 훅에서 LLM-as-judge로 응답을 검증
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. Correct — 실패를 자체 수정한다

도구 호출이 일시적으로 실패하는 일은 흔해요(네트워크, 레이트 리밋 등). `ToolRetryMiddleware`는 지수 백오프로 자동 재시도해요.
또한 외부 evaluator를 별도 노드로 두고 **루프백**시키는 패턴도 자주 쓰여요.

> Correct는 단지 retry가 아니에요. **"같은 실수를 다시 안 하도록 다음 호출에 정보를 더해주는 것"** 까지 포함해요.


**예고편**: 외부 evaluator subgraph 패턴은 **11_Use_Cases/09**에서 자세히 다룹니다. 여기서는 "실패를 자체 수정한다"는 Correct 원칙이 코드로 어떻게 표현되는지 감만 잡으세요.


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain_core.tools import tool

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: Correct 데모: ToolRetryMiddleware로 일시적 실패를 자동 재시도
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### evaluator subgraph로 한 단계 더 나아가기

`ToolRetryMiddleware`가 **도구 단위** 재시도라면, **에이전트 출력 단위** 재시도는 evaluator subgraph로 구현해요.
이 패턴은 사례 A(Anthropic 3-agent harness)에서 다시 만나게 돼요.

```mermaid
flowchart LR
    G["generator<br/>(에이전트)"] --> E["evaluator<br/>(LLM-as-judge)"]
    E -->|"pass"| OUT(["완료"])
    E -->|"fail + 피드백"| G
```


## 7. HITL — 고위험 지점에 사람을 끼워 넣는다

이메일 발송, 결제, DB 삭제처럼 **되돌릴 수 없는 행동**은 모델이 99% 정확해도 위험해요.
LangGraph의 `interrupt()`로 노드 실행을 멈추고, 사람의 결정(`Command(resume=...)`)을 받은 뒤에 진행해요.

> Anthropic — *"For irreversible actions, the cost of a wrong action vastly outweighs the cost of asking."*


**예고편**: `interrupt` 기반 HITL은 **02_LangGraph_Basics/07** 및 **06_Middleware/02**에서 자세히 다룹니다. 여기서는 "고위험 지점에 사람을 끼워 넣는다"는 HITL 원칙이 코드로 어떻게 표현되는지 감만 잡으세요.


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
import uuid

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: HITL 데모: interrupt()로 위험 행동 직전에 사람의 승인을 받기
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 8. 5가지 도구 → 컴포넌트 매핑표

각 원칙을 LangChain V1 / LangGraph / Deep Agents의 어떤 도구로 구현하는지 한눈에 보여줘요. 강의 후반부에서 새로운 컴포넌트를 만날 때마다 이 표를 다시 펼쳐보면 "이건 5가지 도구 중 어디에 해당하지?" 라는 질문에 답할 수 있어요.

| 원칙 | LangChain V1 | LangGraph | Deep Agents |
|---|---|---|---|
| **Constrain** | `ToolCallLimitMiddleware`, `ModelCallLimitMiddleware`, `LLMToolSelectorMiddleware`, `PIIMiddleware` | `RetryPolicy`, conditional edges로 경로 제한 | sub-agent별 도구 화이트리스트, sandbox 권한 격리 |
| **Inform** | `@dynamic_prompt`, `SummarizationMiddleware`, `ContextEditingMiddleware`, `Store` | state 설계, `add_messages` reducer, checkpointer | filesystem 공유 메모, skills, sub-agent 컨텍스트 분리 |
| **Verify** | `@after_model` (LLM-as-judge), structured output 강제, `PIIMiddleware` | validate 노드 + 루프백 엣지 | evaluator sub-agent |
| **Correct** | `ToolRetryMiddleware`, `ModelRetryMiddleware`, `ModelFallbackMiddleware` | `RetryPolicy`, error feedback 루프, `Command(goto=...)` | planner→generator→evaluator 루프, todo 재계획 |
| **HITL** | `HumanInTheLoopMiddleware`(`interrupt_on={tool: ...}`) | `interrupt()` + `Command(resume=...)`, time travel | 위험 도구 호출 전 사용자 컨펌 단계 |

> 같은 원칙을 여러 층에서 **중첩**해서 적용해도 좋아요. 예를 들어 결제 도구는 `Constrain`(권한)으로 사전 차단 + `HITL`(승인)로 마지막 안전망을 둘 수 있어요.


## 9. 산업 사례에서 보는 5가지 도구

이론을 실제 시스템에 어떻게 적용하는지 세 가지 사례를 살펴볼게요.


### 사례 A — Anthropic의 3-agent 구성 (Planner / Generator / Evaluator)

Anthropic은 장기 실행 에이전트의 안정성을 끌어올리기 위해 **하나의 거대한 에이전트 대신 역할이 분리된 3개의 에이전트**로 시스템을 구성했어요.

- **Planner**: 작업을 잘게 쪼개고 다음 단계를 정해요 (Inform + Constrain)
- **Generator**: 실제 도구를 호출하고 결과물을 만들어요 (Constrain + Correct)
- **Evaluator**: 산출물이 기준을 충족하는지 평가하고 통과/재작업을 결정해요 (Verify + Correct 루프)

> 이 패턴은 **`11_Use_Cases/09-Three-Agent-Pattern.ipynb`** 에서 직접 구현해볼 거예요. 그 전단계인 2-node 버전(Generator↔Evaluator)은 **`12_Testing/03-Generator-Evaluator-Verification.ipynb`** 에서 따로 다뤄요 — Planner 없이 평가 분리만 먼저 이해하고 싶을 때 거기부터 보면 좋아요.


### 사례 B — Harness MCP v2의 도구 다이어트 (130 → 11)

초기 Harness MCP 서버는 130개의 도구를 한꺼번에 노출했지만, 모델이 도구를 잘못 선택해서 정확도가 떨어졌어요.
v2에서는 **고수준 작업 11개로 축소**하고, 세부 도구는 sub-agent에게 위임하는 구조로 바꾼 뒤 정확도가 극적으로 올랐어요.

- 핵심 교훈: **도구 수가 늘면 모델의 선택 정확도가 떨어진다.** "할 수 있는 일을 줄이는 것"이 종종 정답이에요. (Constrain의 또 다른 얼굴)

> 이 사례는 **05. Agent Development** 챕터의 08번 레슨(도구 설계)에서 더 깊이 다뤄요.


### 사례 C — Context anxiety와 context reset (Sonnet 4.5)

Sonnet 4.5에서 관찰된 현상이에요. 컨텍스트 윈도우가 차오를수록 모델이 **"시간이 부족하다"고 느껴 서두르고, 품질이 떨어지는** 행동을 보였어요. 이를 *context anxiety*라고 불러요.

해결책은 두 가지예요:
- **Context reset**: 적절한 시점에 대화 히스토리를 요약·압축해서 토큰 압박을 풀어줘요 (`SummarizationMiddleware`)
- **Sub-agent 분리**: 긴 작업은 sub-agent에게 넘겨 **부모 컨텍스트를 깨끗하게** 유지해요

> Inform 원칙은 *"많이 보여주는 것"이 아니라 "필요한 만큼만 보여주는 것"* 임을 보여주는 사례예요.


## 10. 참고: 이후 챕터에서 다시 만나게 돼요

이 5가지 도구는 04~13장에서 middleware, memory, multi-agent, RAG, testing, deployment 같은 다양한 컴포넌트의 모습으로 자연스럽게 다시 등장해요. 지금은 매핑표를 외우려 하지 말고, **새로운 컴포넌트를 볼 때마다 "이건 Constrain/Inform/Verify/Correct/HITL 중 어디에 해당하지?"** 라고 스스로에게 물어보는 습관만 들여도 충분해요.


### 핵심 요약

- 에이전트 = **모델 + 주변 코드**. 모델 외부의 코드가 곧 제품 품질이에요
- **Constrain · Inform · Verify · Correct · HITL** 5가지 도구로 어떤 에이전트 시스템이든 분해해서 이해할 수 있어요
- LangChain V1의 middleware, `@dynamic_prompt`, `@after_model`, `interrupt()` 가 이를 구현하는 핵심 컴포넌트예요
- "에이전트가 실수했다"는 모델 탓이 아니라 **주변 코드를 한 겹 덧댈 신호**예요
- 업계에서는 이 사고를 *harness engineering* 이라 부르기도 하지만, 용어가 바뀌어도 5가지 도구 자체는 살아남아요

### 다음 노트북 예고

다음 `04_LangGraph_Advanced/01-Functional-API.ipynb`에서는 같은 신뢰성 원칙을 **그래프 실행 단위**로 구현하는 방법을 배워요. `@entrypoint`와 `@task`로 작업을 나누고, 체크포인트·재시도·interrupt를 더 작은 실행 단위에 붙이는 것이 핵심이에요. 이후 12_Testing과 13_Deployment Capstone에서는 여기서 배운 5가지 도구를 평가·운영 관점으로 다시 확인해요.

### 더 읽을거리
  - [Anthropic: Building Effective Agents](https://www.anthropic.com/research/building-effective-agents)
  - [HumanLayer: 12-Factor Agents](https://github.com/humanlayer/12-factor-agents)
  - LangChain V1 middleware 공식 문서
